In [2]:
import os

import logging
from datetime import datetime
from pathlib import Path
from tqdm import tqdm

import numpy as np

import tvm

from tvm import relay
from tvm.relay.backend import Executor
from tvm.contrib import utils
from tvm import meta_schedule as ms
from tvm.driver import tvmc
from tvm.meta_schedule.runner import EvaluatorConfig
from tvm.meta_schedule.logging import get_logger
from tvm import transform
from tvm.contrib.micro.meta_schedule.local_builder_micro import get_local_builder_micro
from tvm.contrib.micro.meta_schedule.rpc_runner_micro import get_rpc_runner_micro

from model_info import get_model_info


logging.basicConfig(level=logging.ERROR)
get_logger("xgb_model").setLevel(logging.ERROR)

DIR = Path("./").parent.resolve()
BASE_DIR = DIR.parent

GCC_PREFIX = os.environ.get("GCC_PREFIX", "/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/deps/install/riscv_gcc_rv32")
GCC_NAME = os.environ.get("GCC_NAME", "riscv32-unknown-elf")
LLVM_DIR = os.environ.get("LLVM_DIR", "/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/deps/install/llvm")
ETISS_TEMPLATE = os.environ.get("ETISS_TEMPLATE", "/nfs/TUEIEDAscratch/ge85zic/tuning")
ETISS_SCRIPT = os.environ.get("ETISS_SCRIPT", "/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/deps/install/etiss/bin/run_helper.sh")
PLATFORM = os.path.join(ETISS_TEMPLATE, "template_project")

path = "/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/models/resnet/resnet.tflite"
tir_path = "/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/temp/sessions/941/runs/0/default.tir"



## Examine each part

In [22]:

ALTER_OP = True
TOOLCHAIN = "gcc"
TARGET = "c -num-cores 1"
NUM_TRIALS_PER_ITER, MAX_TRIALS_PER_TASK, MAX_TRIALS_GLOBAL = (5, 50, 1000000)
TASK_FILTER = [0, 1]
MODULE_EQUALITY = "ignore-ndarray"
TRANSFORM_LAYOUT = False

OPTIONS = {
    "verbose": True,
    "quiet": True,
    "gcc_prefix": str(GCC_PREFIX),
    "gcc_name": GCC_NAME,
    "llvm_dir": str(LLVM_DIR),
    "etiss_script": str(ETISS_SCRIPT),
    "etiss_args": "",
    "arch": "rv32gc_zicsr_zifencei",
    "abi": "ilp32d",
    "cpu_arch": "RV32IMACFD",
    "cpu_freq": 100000000,
    "toolchain": TOOLCHAIN,
}

MS_DISPATCH = 1  # silent?
# MS_DISPATCH = 2  # verbose
# MS_DISPATCH = ?  # error
SKIP_TUNING = False

In [ ]:


def load_model(model):
    def _load_model(path, shape_dict):
        model = tvmc.load(
            str(path),
            shape_dict=shape_dict,
        )
        mod = model.mod
        params = model.params
        return mod, params

    model_info = get_model_info(model)
    shape_dict = {t.name: t.shape for t in model_info.in_tensors}
    assert len(model_info.in_tensors) == 1
    input_name, input_shape = list(shape_dict.items())[0]
    input_dtype = model_info.in_tensors[0].dtype
    data_sample = np.random.rand(*input_shape).astype(input_dtype)
    mod, params = _load_model(model, shape_dict)
    return mod, params, input_name, input_shape, input_dtype, data_sample

mod, params, input_name, input_shape, input_dtype, data_sample = load_model(path)
# mod is the relay code
# params is the weights
# input name 

In [24]:
opt_level = 3
pass_config = {
    "tir.disable_vectorize": True,
}
disabled_pass = []
if not ALTER_OP:
    disabled_pass += ["AlterOpLayout"]

KEEP = True
if KEEP:
    base_dir = Path("/tmp/base2")
    now = datetime.now()
    ts = now.strftime("%Y%m%dT%H%M%S")

    label = ts
    work_dir_path = base_dir / label
else:
    work_dir = utils.tempdir()
    work_dir_path = work_dir.path
print("work_dir_path", work_dir_path)
# mod, params, input_name, input_shape, input_dtype, data_sample = load_model(model)

if TRANSFORM_LAYOUT:
    with tvm.transform.PassContext(
        opt_level=opt_level,
        config=pass_config,
        disabled_pass=disabled_pass,
    ):
        desired_layouts = {"qnn.conv2d": ["NCHW", "default"]}

        # Convert the layout of the graph where possible.
        seq = transform.Sequential(
            [
                relay.transform.RemoveUnusedFunctions(),
                relay.transform.ConvertLayout(desired_layouts),
                relay.transform.FoldConstant(),
            ]
        )
        mod = seq(mod)
        print(mod)




work_dir_path /tmp/base2/20250818T101719


In [ ]:
link_params = True


# In essence, relay.backend.Runtime("crt", {"system-lib": True}) creates a configuration for building a compact, self-contained executable that can run on a device using a minimal C-based runtime.
runtime = relay.backend.Runtime("crt", {"system-lib": True}) # C runtime


# In summary, Executor("aot", {"link-params": link_params}) configures TVM to compile the model into a standalone executable that includes the model's weights and biases, making it ready for deployment to a target device.
executor = Executor("aot", {"link-params": link_params}) # link model weights
# This line is necessary for link-params to take effect during
# task extraction and relay.build(...).
mod = mod.with_attr("executor", executor)

builder = get_local_builder_micro() # Annotates the relay code in mod

In [30]:
def get_tuning_config():
    def _get_sch_rules():
        structure = "SR"
        return [
            ms.schedule_rule.ApplyCustomRule(),
            ms.schedule_rule.InlineConstantScalars(),
            ms.schedule_rule.AutoInline(
                into_producer=False,
                into_consumer=True,
                inline_const_tensor=True,
                disallow_if_then_else=True,
                require_injective=True,
                require_ordered=True,
                disallow_op=["tir.exp"],
            ),
            ms.schedule_rule.MultiLevelTiling(
                structure="SSRSRS",
                tile_binds=None,
                max_innermost_factor=64,
                vector_load_lens=None,
                reuse_read=None,
                reuse_write=ms.schedule_rule.ReuseType(
                    req="may",
                    levels=[1, 2],
                    scope="global",
                ),
            ),
            ms.schedule_rule.ParallelizeVectorizeUnroll(
                max_jobs_per_core=-1,  # disable parallelize
                max_vectorize_extent=-1,  # disable vectorize
                unroll_max_steps=[0, 2, 4, 8, 16, 32, 64],
                unroll_explicit=True,
                # unroll_explicit=False,
            ),
            ms.schedule_rule.RandomComputeLocation(),
        ]

    def _get_postprocs():
        return [
            ms.postproc.DisallowDynamicLoop(),
            ms.postproc.RewriteParallelVectorizeUnroll(),
            ms.postproc.RewriteReductionBlock(),
        ]

    def _get_mutator_probs():
        return {
            ms.mutator.MutateTileSize(): 0.9,
            ms.mutator.MutateComputeLocation(): 0.05,
            ms.mutator.MutateUnroll(): 0.03,
            # ms.mutator.Parallel(): 0.02,
        }

    sch_rules = _get_sch_rules()
    postprocs = _get_postprocs()
    mutator_probs = _get_mutator_probs()
    return sch_rules, postprocs, mutator_probs


if not SKIP_TUNING:
    sch_rules, postprocs, mutator_probs = get_tuning_config()
    space = ms.space_generator.PostOrderApply(
        sch_rules=sch_rules,
        postprocs=postprocs,
        mutator_probs=mutator_probs,
    )
    strategy = "evolutionary"
    evaluator_config = EvaluatorConfig(
        number=1,
        repeat=1,
        min_repeat_ms=0,
        enable_cpu_cache_flush=False,
    )
    extractor = ms.feature_extractor.PerStoreFeature()
    num_warmup_samples = 10
    cost_model = ms.cost_model.XGBModel(extractor=extractor, num_warmup_samples=num_warmup_samples)

In [31]:
tasks, task_weights = ms.relay_integration.extracted_tasks_to_tune_contexts(
                        extracted_tasks=ms.relay_integration.extract_tasks(
                            mod,
                            TARGET,
                            params,
                            opt_level=opt_level,
                            module_equality=MODULE_EQUALITY,
                            pass_config=pass_config,
                            disabled_pass=disabled_pass,
                        ),
                        work_dir=str(work_dir_path),
                        space=space,
                        strategy=strategy,
                        num_tuning_cores=1,
                    )

2025-08-18 10:22:34 [INFO] Logging directory: /tmp/base2/20250818T101719/logs


In [38]:
pass_config

{'tir.disable_vectorize': True}

In [15]:
import tvm
from tvm.script import tir as T

# Step 1: Define the TIR code as a string
tir_source_code = """




# from tvm.script import tir as T

@T.prim_func
def tvmgen_default_fused_reshape_cast_subtract(p0: T.Buffer((1, 640), "int8"), T_subtract: T.Buffer((1, 640), "int16")):
    T.func_attr({"from_legacy_te_schedule": T.bool(True), "tir.noalias": T.bool(True)})
    for ax1_outer, ax1_inner in T.grid(40, 16):
        T_subtract_1 = T.Buffer((640,), "int16", data=T_subtract.data)
        p0_1 = T.Buffer((640,), "int8", data=p0.data)
        T_subtract_1[ax1_outer * 16 + ax1_inner] = T.Cast("int16", p0_1[ax1_outer * 16 + ax1_inner]) - T.int16(89)
# from tvm.script import tir as T

@T.prim_func
def tvmgen_default_fused_nn_contrib_dense_pack_add_fixed_point_multiply_add_clip_cast(p0: T.Buffer((1, 640), "int16"), T_cast: T.Buffer((1, 128), "int8"), fused_nn_contrib_dense_pack_constant: T.Buffer((1, 128), "int32"), fused_constant: T.Buffer((16, 640, 8), "int16")):
    T.func_attr({"from_legacy_te_schedule": T.bool(True), "tir.noalias": T.bool(True)})
    for ax1_outer_ax0_outer_fused in T.parallel(4):
        compute = T.allocate([32], "int32", "global")
        compute_global = T.allocate([8], "int32", "global")
        compute_1 = T.Buffer((32,), "int32", data=compute)
        for y_inner_outer_x_inner_outer_fused in range(4):
            compute_global_1 = T.Buffer((8,), "int32", data=compute_global, align=32)
            for x_c_init in range(8):
                compute_global_1[x_c_init] = 0
            for k_outer, x_c in T.grid(640, 8):
                p0_1 = T.Buffer((640,), "int16", data=p0.data)
                fused_constant_1 = T.Buffer((81920,), "int16", data=fused_constant.data)
                compute_global_1[x_c] = compute_global_1[x_c] + T.Cast("int32", p0_1[k_outer]) * T.Cast("int32", fused_constant_1[ax1_outer_ax0_outer_fused * 20480 + y_inner_outer_x_inner_outer_fused * 5120 + k_outer * 8 + x_c])
            for x_inner_inner in range(8):
                compute_1[y_inner_outer_x_inner_outer_fused * 8 + x_inner_inner] = compute_global_1[x_inner_inner]
        for ax1_inner_outer, ax1_inner_inner in T.grid(4, 8):
            T_cast_1 = T.Buffer((128,), "int8", data=T_cast.data)
            fused_nn_contrib_dense_pack_constant_1 = T.Buffer((128,), "int32", data=fused_nn_contrib_dense_pack_constant.data)
            T_cast_1[ax1_outer_ax0_outer_fused * 32 + ax1_inner_outer * 8 + ax1_inner_inner] = T.Cast("int8", T.max(T.min(T.q_multiply_shift(compute_1[ax1_inner_outer * 8 + ax1_inner_inner] + fused_nn_contrib_dense_pack_constant_1[ax1_outer_ax0_outer_fused * 32 + ax1_inner_outer * 8 + ax1_inner_inner], 1638001653, 31, -8) - 128, 127), -128))
"""
funct = [x for x in tir_source_code.split("# from tvm.script import tir as T") if x.strip() ]
funct
# Step 2 & 3: Parse the string to an IRModule object
parsed_module = [tvm.script.from_source(tir_source_code) for tir_source_code in funct]
parsed_module

# # Now, 'parsed_module' is an IRModule object
# print(type(parsed_module))
# print(parsed_module.script()) # Print the parsed module

[# from tvm.script import tir as T
 
 @T.prim_func
 def tvmgen_default_fused_reshape_cast_subtract(p0: T.Buffer((1, 640), "int8"), T_subtract: T.Buffer((1, 640), "int16")):
     T.func_attr({"from_legacy_te_schedule": T.bool(True), "tir.noalias": T.bool(True)})
     for ax1_outer, ax1_inner in T.grid(40, 16):
         T_subtract_1 = T.Buffer((640,), "int16", data=T_subtract.data)
         p0_1 = T.Buffer((640,), "int8", data=p0.data)
         T_subtract_1[ax1_outer * 16 + ax1_inner] = T.Cast("int16", p0_1[ax1_outer * 16 + ax1_inner]) - T.int16(89),
 # from tvm.script import tir as T
 
 @T.prim_func
 def tvmgen_default_fused_nn_contrib_dense_pack_add_fixed_point_multiply_add_clip_cast(p0: T.Buffer((1, 640), "int16"), T_cast: T.Buffer((1, 128), "int8"), fused_nn_contrib_dense_pack_constant: T.Buffer((1, 128), "int32"), fused_constant: T.Buffer((16, 640, 8), "int16")):
     T.func_attr({"from_legacy_te_schedule": T.bool(True), "tir.noalias": T.bool(True)})
     for ax1_outer_ax0_outer_fus

In [ ]:
def load_tir(tir_path):
    with open(tir_path, "r") as f:
        content = f.read()

    # As tvm.script.from_source is unable to handle multiple Primfunc in a file
    funct = [x for x in content.split("# from tvm.script import tir as T") if x.strip() ]
    objs = [tvm.script.from_source(tir_source_code) for tir_source_code in funct]

    ret=[]
    for obj in objs:
        if isinstance(obj, tvm.tir.PrimFunc):
            default_name = "main"
            obj = tvm.IRModule({default_name: obj})
        assert isinstance(obj, tvm.IRModule)
        ret.append(obj)
    return ret

tir_path = "/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/temp/sessions/941/runs/0/default.tir"
tasks = load_tir(tir_path)
# from .parser import parse as from_source
# from ._core import parse

## Entire code

In [7]:

def load_model(model):
    def _load_model(path, shape_dict):
        model = tvmc.load(
            str(path),
            shape_dict=shape_dict,
        )
        mod = model.mod
        params = model.params
        return mod, params

    model_info = get_model_info(model)
    shape_dict = {t.name: t.shape for t in model_info.in_tensors}
    assert len(model_info.in_tensors) == 1
    input_name, input_shape = list(shape_dict.items())[0]
    input_dtype = model_info.in_tensors[0].dtype
    data_sample = np.random.rand(*input_shape).astype(input_dtype)
    mod, params = _load_model(model, shape_dict)
    return mod, params, input_name, input_shape, input_dtype, data_sample


def get_tuning_config():
    def _get_sch_rules():
        structure = "SR"
        return [
            ms.schedule_rule.ApplyCustomRule(),
            ms.schedule_rule.InlineConstantScalars(),
            ms.schedule_rule.AutoInline(
                into_producer=False,
                into_consumer=True,
                inline_const_tensor=True,
                disallow_if_then_else=True,
                require_injective=True,
                require_ordered=True,
                disallow_op=["tir.exp"],
            ),
            ms.schedule_rule.MultiLevelTiling(
                structure="SSRSRS",
                tile_binds=None,
                max_innermost_factor=64,
                vector_load_lens=None,
                reuse_read=None,
                reuse_write=ms.schedule_rule.ReuseType(
                    req="may",
                    levels=[1, 2],
                    scope="global",
                ),
            ),
            ms.schedule_rule.ParallelizeVectorizeUnroll(
                max_jobs_per_core=-1,  # disable parallelize
                max_vectorize_extent=-1,  # disable vectorize
                unroll_max_steps=[0, 2, 4, 8, 16, 32, 64],
                unroll_explicit=True,
                # unroll_explicit=False,
            ),
            ms.schedule_rule.RandomComputeLocation(),
        ]

    def _get_postprocs():
        return [
            ms.postproc.DisallowDynamicLoop(),
            ms.postproc.RewriteParallelVectorizeUnroll(),
            ms.postproc.RewriteReductionBlock(),
        ]

    def _get_mutator_probs():
        return {
            ms.mutator.MutateTileSize(): 0.9,
            ms.mutator.MutateComputeLocation(): 0.05,
            ms.mutator.MutateUnroll(): 0.03,
            # ms.mutator.Parallel(): 0.02,
        }

    sch_rules = _get_sch_rules()
    postprocs = _get_postprocs()
    mutator_probs = _get_mutator_probs()
    return sch_rules, postprocs, mutator_probs


def _schedule_dummy():

    def schedule_fn(sch, block=None) -> bool:
        return True

    return schedule_fn


ALTER_OP = True
TOOLCHAIN = "gcc"
TARGET = "c -num-cores 1"
NUM_TRIALS_PER_ITER, MAX_TRIALS_PER_TASK, MAX_TRIALS_GLOBAL = (5, 50, 1000000)
TASK_FILTER = [0, 1]
MODULE_EQUALITY = "ignore-ndarray"
TRANSFORM_LAYOUT = False

OPTIONS = {
    "verbose": True,
    "quiet": True,
    "gcc_prefix": str(GCC_PREFIX),
    "gcc_name": GCC_NAME,
    "llvm_dir": str(LLVM_DIR),
    "etiss_script": str(ETISS_SCRIPT),
    "etiss_args": "",
    "arch": "rv32gc_zicsr_zifencei",
    "abi": "ilp32d",
    "cpu_arch": "RV32IMACFD",
    "cpu_freq": 100000000,
    "toolchain": TOOLCHAIN,
}

MS_DISPATCH = 1  # silent?
# MS_DISPATCH = 2  # verbose
# MS_DISPATCH = ?  # error
SKIP_TUNING = False

def test_micro_tuning_with_meta_schedule(platform, alter_op, target, num_trials_per_iter, max_trials_per_task, max_trials_global, module_equality, model, transform_layout, options, task_filter):
    opt_level = 3
    pass_config = {
        "tir.disable_vectorize": True,
    }
    disabled_pass = []
    if not alter_op:
        disabled_pass += ["AlterOpLayout"]

    KEEP = True
    if KEEP:
        base_dir = Path("/tmp/base2")
        now = datetime.now()
        ts = now.strftime("%Y%m%dT%H%M%S")

        label = ts
        work_dir_path = base_dir / label
    else:
        work_dir = utils.tempdir()
        work_dir_path = work_dir.path
    print("work_dir_path", work_dir_path)
    mod, params, input_name, input_shape, input_dtype, data_sample = load_model(model)

    if transform_layout:
        with tvm.transform.PassContext(
            opt_level=opt_level,
            config=pass_config,
            disabled_pass=disabled_pass,
        ):
            desired_layouts = {"qnn.conv2d": ["NCHW", "default"]}

            # Convert the layout of the graph where possible.
            seq = transform.Sequential(
                [
                    relay.transform.RemoveUnusedFunctions(),
                    relay.transform.ConvertLayout(desired_layouts),
                    relay.transform.FoldConstant(),
                ]
            )
            mod = seq(mod)

    link_params = True

    runtime = relay.backend.Runtime("crt", {"system-lib": True})
    executor = Executor("aot", {"link-params": link_params})
    # This line is necessary for link-params to take effect during
    # task extraction and relay.build(...).
    mod = mod.with_attr("executor", executor)

    builder = get_local_builder_micro()

    with ms.Profiler() as profiler:
        if not SKIP_TUNING:
            sch_rules, postprocs, mutator_probs = get_tuning_config()
            space = ms.space_generator.PostOrderApply(
                sch_rules=sch_rules,
                postprocs=postprocs,
                mutator_probs=mutator_probs,
            )
            strategy = "evolutionary"
            evaluator_config = EvaluatorConfig(
                number=1,
                repeat=1,
                min_repeat_ms=0,
                enable_cpu_cache_flush=False,
            )
            extractor = ms.feature_extractor.PerStoreFeature()
            num_warmup_samples = 10
            cost_model = ms.cost_model.XGBModel(extractor=extractor, num_warmup_samples=num_warmup_samples)
            # micro_rpc_workers = num_trials_per_iter
            with get_rpc_runner_micro(
                platform=platform, options=options, session_timeout_sec=120, evaluator_config=evaluator_config,
                # serial_numbers=["micro"] * micro_rpc_workers,
                tracker_host="127.0.0.1",
                tracker_port=9020,
                # max_workers=micro_rpc_workers,
                rpc_timeout_sec=10,

            ) as runner:
                if max_trials_global > 0:
                    tasks, task_weights = ms.relay_integration.extracted_tasks_to_tune_contexts(
                        extracted_tasks=ms.relay_integration.extract_tasks(
                            mod,
                            target,
                            params,
                            opt_level=opt_level,
                            module_equality=module_equality,
                            pass_config=pass_config,
                            disabled_pass=disabled_pass,
                        ),
                        work_dir=str(work_dir_path),
                        space=space,
                        strategy=strategy,
                        num_tuning_cores=1,
                    )
                    if task_filter is not None:
                        assert isinstance(task_filter, list)
                        assert len(task_filter) > 0
                        tasks = [tasks[i] for i in task_filter]
                        task_weights = [task_weights[i] for i in task_filter]
                    pass_config = dict(pass_config)
                    with transform.PassContext(
                        opt_level=opt_level,
                        config=pass_config,
                        disabled_pass=disabled_pass,
                    ):
                        db: ms.Database = ms.tune.tune_tasks(
                            tasks=tasks,
                            task_weights=task_weights,
                            work_dir=str(work_dir_path),
                            max_trials_global=max_trials_global,
                            max_trials_per_task=max_trials_per_task,
                            num_trials_per_iter=num_trials_per_iter,
                            builder=builder,
                            runner=runner,
                            cost_model=cost_model,
                            module_equality=module_equality,
                        )
                else:
                    # db = ms.database.MemoryDatabase()
                    db = ms.database.ScheduleFnDatabase(
                        _schedule_dummy()
                    )

            #  Build model using meta_schedule logs
            ms_mod: tvm.runtime.Module = ms.relay_integration.compile_relay(
                database=db,
                mod=mod,
                target=target,
                params=params,
                pass_config=MappingProxyType(
                    {
                        **pass_config,
                        "relay.backend.use_meta_schedule": True,
                        "relay.backend.tir_converter": "default",
                        "relay.backend.use_meta_schedule_dispatch": MS_DISPATCH,
                    }
                ),
                disabled_pass=disabled_pass,
                executor=executor,
                runtime=runtime,
            )
    # print("tasks[0]", tasks[0], dir(tasks[0]))
    # print("tasks[0]", tasks[0].mod)
    print(profiler.table())
    print("cost_model", cost_model, dir(cost_model))
    saved_model_path = work_dir_path / "cost_model.tar"
    # random_state = model.extractor.random_state
    cost_model.save(str(saved_model_path))
    cost_model.load(str(saved_model_path))
    cost_model.num_warmup_samples = 1  # Do not get random predictions
    # model.extractor.random_state = random_state
    # candidate = MeasureCandidate(Schedule(FullModule), [])
    dummy_preds = []
    record_preds = []
    for i in range(len(tasks)):
        tune_ctx = tasks[i]
        print("tune_ctx", tune_ctx, dir(tune_ctx))
        sched = tir.Schedule(tune_ctx.mod)
        print("sched", sched)
        # dummy_candidate = _make_candidate(sched)
        dummy_candidate = ms.MeasureCandidate(sch=sched, args_info=[])
        print("dummy_candidate", dummy_candidate)
        (dummy_feature,) = extractor.extract_from(
            tune_ctx,
            candidates=[dummy_candidate],
        )
        print("dummy_feature", dummy_feature, dir(dummy_feature))
        dummy_predictions = cost_model.predict(tune_ctx, [dummy_candidate])
        dummy_preds.append(dummy_predictions[0])
        print("dummy_predictions", dummy_predictions)
        workload = db.commit_workload(tasks[i].mod)
        records = db.get_top_k(workload, 3)
        print("records", records, len(records))
        if len(records) == 0:
            continue
        record = records[0]
        print("record", record, dir(record))
        db.commit_tuning_record(record)
        record_trace = record.trace
        print("record_trace", record_trace, dir(record_trace))
        record_sched = tir.Schedule(record.workload.mod)
        print("record_sched_init", record_sched, dir(record_sched))
        record_trace.apply_to_schedule(record_sched, remove_postproc=False)
        print("record_sched", record_sched, dir(record_sched))
        record_candidate = ms.MeasureCandidate(sch=record_sched, args_info=[])
        print("record_candidate", record_candidate)
        (record_feature,) = extractor.extract_from(
            tune_ctx,
            candidates=[record_candidate],
        )
        print("record_feature", record_feature, dir(record_feature))
        record_predictions = cost_model.predict(tune_ctx, [record_candidate])
        print("record_predictions", record_predictions)
        # assert len(record_predictions) == 1
        record_preds.append(record_predictions[0])
    print("dummy_preds", dummy_preds)
    sorted_dummy_idxs = list(np.argsort(dummy_preds))
    print("sorted_dummy_idxs", sorted_dummy_idxs)
    sorted_dummy_preds = [dummy_preds[i] for i in sorted_dummy_idxs]
    print("sorted_dummy_preds", sorted_dummy_preds)
    dummy_preds_sum = sum(dummy_preds)
    print("dummy_preds_sum", dummy_preds_sum)
    print("record_preds", record_preds)
    sorted_record_idxs = list(np.argsort(record_preds))
    print("sorted_record_idxs", sorted_record_idxs)
    sorted_record_preds = [record_preds[i] for i in sorted_record_idxs]
    print("sorted_record_preds", sorted_record_preds)
    record_preds_sum = sum(record_preds)
    print("record_preds_sum", record_preds_sum)
    # TODO: weighted sum!
    input("!!!")
    non_ms_mod: tvm.runtime.Module = ms.relay_integration.compile_relay(
        None,
        mod=mod,
        target=target,
        params=params,
        pass_config=MappingProxyType(
            {
                **pass_config,
                "relay.backend.use_meta_schedule_dispatch": MS_DISPATCH,
            }
        ),
        disabled_pass=disabled_pass,
        executor=executor,
        runtime=runtime,
    )

    if not SKIP_TUNING:
        # TUNED
        # TODO: wrap in helper
        project = tvm.micro.generate_project(
            str(platform),
            ms_mod,
            str(work_dir_path / "project"),
            options=options,
        )
        project.build()
        project.flash()
        with tvm.micro.Session(project.transport()) as session:
            aot_executor = tvm.runtime.executor.aot_executor.AotModule(session.create_aot_executor())
            result = aot_executor.module.time_evaluator("run", session.device, number=1)()
            print("result", result)
            print("mean: ", result.mean)

    # UNTUNED
    project = tvm.micro.generate_project(
        str(platform),
        non_ms_mod,
        str(work_dir_path / "project2"),
        options=options,
    )
    project.build()
    project.flash()
    with tvm.micro.Session(project.transport()) as session:
        aot_executor = tvm.runtime.executor.aot_executor.AotModule(session.create_aot_executor())
        result2 = aot_executor.module.time_evaluator("run", session.device, number=1)()
        print("result2", result2)
        print("mean2:", result2.mean)
    if not SKIP_TUNING:
        rel = result.mean / result2.mean
        print("rel:  ", rel)



# MODEL = "/work/git/mlonmcu/mlonmcu/workspace_default/models/resnet/resnet.tflite"
# assert len(sys.argv) == 2, "Usage: micro_ms_cost_model_etiss.py MODEL_PATH"


In [8]:
MODEL = "/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/models/resnet/resnet.tflite"
test_micro_tuning_with_meta_schedule(PLATFORM, ALTER_OP, TARGET, NUM_TRIALS_PER_ITER, MAX_TRIALS_PER_TASK, MAX_TRIALS_GLOBAL, MODULE_EQUALITY, MODEL, TRANSFORM_LAYOUT, OPTIONS, TASK_FILTER)


work_dir_path /tmp/base2/20250817T174543
2025-08-17 17:45:54 [INFO] Logging directory: /tmp/base2/20250817T174543/logs
2025-08-17 17:46:05 [INFO] [task_scheduler.cc:164] Initializing Task #0: "fused_nn_conv2d_add_fixed_point_multiply_per_axis_add_clip_cast_subtract"
2025-08-17 17:46:05 [INFO] [task_scheduler.cc:164] Initializing Task #1: "fused_nn_conv2d_add_fixed_point_multiply_per_axis_add_clip_subtract_fixed_point__eb606f94f03ebac6_"


,Name,FLOP,Weight,Speed (GFLOPS),Latency (us),Weighted Latency (us),Trials,Done
0,fused_nn_conv2d_add_fixed_point_multiply_per_axis_add_clip_cast_subtract,2379776,1,N/A,N/A,N/A,0,
1,fused_nn_conv2d_add_fixed_point_multiply_per_axis_add_clip_subtract_fixed_point__eb606f94f03ebac6_,4743168,1,N/A,N/A,N/A,0,



Total trials: 0
Total latency (us): 0
2025-08-17 17:46:05 [DEBUG] [task_scheduler.cc:323] 
 ID |                                                                                               Name |    FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
  0 |                           fused_nn_conv2d_add_fixed_point_multiply_per_axis_add_clip_cast_subtract | 2379776 |      1 |            N/A |          N/A |                   N/A |      0 |      
  1 | fused_nn_conv2d_add_fixed_point_multiply_per_axis_add_clip_subtract_fixed_point__eb606f94f03ebac6_ | 4743168 |      1 |            N/A |          N/A |                   N/A |      0 |      
------------------------------------------------------------------------------------------------------------------------

,Name,FLOP,Weight,Speed (GFLOPS),Latency (us),Weighted Latency (us),Trials,Done
0,fused_nn_conv2d_add_fixed_point_multiply_per_axis_add_clip_cast_subtract,2379776,1,N/A,N/A,N/A,5,
1,fused_nn_conv2d_add_fixed_point_multiply_per_axis_add_clip_subtract_fixed_point__eb606f94f03ebac6_,4743168,1,N/A,N/A,N/A,0,



Total trials: 0
Total latency (us): 0

2025-08-17 17:49:30 [DEBUG] [task_scheduler.cc:323] 
 ID |                                                                                               Name |    FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
  0 |                           fused_nn_conv2d_add_fixed_point_multiply_per_axis_add_clip_cast_subtract | 2379776 |      1 |            N/A |          N/A |                   N/A |      5 |      
  1 | fused_nn_conv2d_add_fixed_point_multiply_per_axis_add_clip_subtract_fixed_point__eb606f94f03ebac6_ | 4743168 |      1 |            N/A |          N/A |                   N/A |      0 |      
-----------------------------------------------------------------------------------------------------------------------

,Name,FLOP,Weight,Speed (GFLOPS),Latency (us),Weighted Latency (us),Trials,Done
0,fused_nn_conv2d_add_fixed_point_multiply_per_axis_add_clip_cast_subtract,2379776,1,N/A,N/A,N/A,5,
1,fused_nn_conv2d_add_fixed_point_multiply_per_axis_add_clip_subtract_fixed_point__eb606f94f03ebac6_,4743168,1,N/A,N/A,N/A,5,


2025-08-17 17:50:55 [DEBUG] [task_scheduler.cc:323] 
 ID |                                                                                               Name |    FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
  0 |                           fused_nn_conv2d_add_fixed_point_multiply_per_axis_add_clip_cast_subtract | 2379776 |      1 |            N/A |          N/A |                   N/A |      5 |      
  1 | fused_nn_conv2d_add_fixed_point_multiply_per_axis_add_clip_subtract_fixed_point__eb606f94f03ebac6_ | 4743168 |      1 |            N/A |          N/A |                   N/A |      5 |      
---------------------------------------------------------------------------------------------------------------------------------------------------------------

: 

## Standalone_MS_cost

In [3]:
from standalone_ms_cost_model import *

In [20]:
import tvm

def benchmark_mod(ir_module):
    """
    Builds IRModule and returns the mean time taken to execute the function in milliseconds.
    """
    func = ir_module["main"]
    test_inputs = []
    func_name = func.attrs["global_symbol"]
    
    # The function's buffer_map holds the key-value pair of handle to Buffer object
    for _, buffer_obj in func.buffer_map.items():
    
        shape = tuple(int(s) for s in buffer_obj.shape)
        dtype = buffer_obj.dtype
        test_inputs.append(tvm.nd.array(np.random.rand(*shape).astype(dtype)))
    
    lib = tvm.build(ir_module, target="llvm")
    f_time = lib.time_evaluator(func_name, tvm.cpu())

    return f_time(*test_inputs).mean * 1000
    

    


In [5]:
mods = load_tir(tir_path)

In [6]:
for mod in mods:
    a=mod
    break

In [9]:
build_obj.export_library("benchmark_mod.s")

In [21]:
import glob
from tqdm import tqdm

def generate_samples_from_session(session_path):
    """
    Generates samples from a session path.
    """
    tir_files = glob.glob(os.path.join(session_path, "default.tir*"))
    if not tir_files:
        raise ValueError(f"No TIR files found in the session path: {session_path}")
    mods = []
    for tir_file in tir_files:
        mods.extend(load_tir(tir_file))
    return [(mod, benchmark_mod(mod)) for mod in tqdm(mods)]

session_path = "/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/temp/sessions/941/runs/0"
samples_resnet = generate_samples_from_session(session_path)

error: realize() missing 1 required positional argument: 'storage_scope'
 --> <str>:6:5
   |  
 6 |      T.realize(T_subtract[0:1, 0:640])
   |      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
error: realize() missing 1 required positional argument: 'storage_scope'
 --> <str>:6:5
   |  
 6 |      T.realize(T_cast[0:1, 0:128])
   |      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
error: realize() missing 1 required positional argument: 'storage_scope'
 --> <str>:6:5
   |  
 6 |      T.realize(T_subtract[0:1, 0:128])
   |      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
error: realize() missing 1 required positional argument: 'storage_scope'
 --> <str>:6:5
   |  
 6 |      T.realize(T_cast[0:1, 0:128])
   |      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
error: realize() missing 1 required positional argument: 'storage_scope'
 --> <str>:6:5
   |  
 6 |      T.realize(T_cast[0:1, 0:128])
   |      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
error: realize() missing 1 required positional argument: 'storage_scope'
 --> <str>:6:5
   |  
 6 |      T.re

Error parsing TIR source code, trying workaround for T.realize
Successfully replaced T.realize with global realization
Error parsing TIR source code, trying workaround for T.realize
Successfully replaced T.realize with global realization
Error parsing TIR source code, trying workaround for T.realize
Successfully replaced T.realize with global realization
Error parsing TIR source code, trying workaround for T.realize
Successfully replaced T.realize with global realization
Error parsing TIR source code, trying workaround for T.realize
Successfully replaced T.realize with global realization
Error parsing TIR source code, trying workaround for T.realize
Successfully replaced T.realize with global realization
Error parsing TIR source code, trying workaround for T.realize
Successfully replaced T.realize with global realization


error: realize() missing 1 required positional argument: 'storage_scope'
 --> <str>:6:5
   |  
 6 |      T.realize(T_subtract[0:1, 0:8])
   |      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
error: realize() missing 1 required positional argument: 'storage_scope'
 --> <str>:6:5
   |  
 6 |      T.realize(T_cast[0:1, 0:128])
   |      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
error: realize() missing 1 required positional argument: 'storage_scope'
 --> <str>:6:5
   |  
 6 |      T.realize(T_cast[0:1, 0:128])
   |      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
error: realize() missing 1 required positional argument: 'storage_scope'
 --> <str>:6:5
   |  
 6 |      T.realize(T_cast[0:1, 0:128])
   |      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
error: realize() missing 1 required positional argument: 'storage_scope'
 --> <str>:6:5
   |  
 6 |      T.realize(T_cast[0:1, 0:128])
   |      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
error: realize() missing 1 required positional argument: 'storage_scope'
 --> <str>:6:5
   |  
 6 |      T.realize(T_cast

Error parsing TIR source code, trying workaround for T.realize
Successfully replaced T.realize with global realization
Error parsing TIR source code, trying workaround for T.realize
Successfully replaced T.realize with global realization
Error parsing TIR source code, trying workaround for T.realize
Successfully replaced T.realize with global realization
Error parsing TIR source code, trying workaround for T.realize
Successfully replaced T.realize with global realization
Error parsing TIR source code, trying workaround for T.realize
Successfully replaced T.realize with global realization
Error parsing TIR source code, trying workaround for T.realize
Successfully replaced T.realize with global realization


100%|██████████| 65/65 [00:08<00:00,  7.94it/s]


In [17]:
time = [time for _, time in samples_resnet]
sum(time)

0.7152859999999998